### Rominos – Lösung Schritt für Schritt

*38. Bundeswettbewerb Informatik (2019), 1. Runde, Aufgabe 5*

Ruth hat zum Geburtstag eine Schachtel mit zwölf Figuren geschenkt bekommen: Sie heißen Pentominos und sind jeweils aus fünf gleich großen Quadraten gebildet. Es gibt genau zwölf verschiedene Pentominos, wie in Ruths Schachtel.

Ruth stellt fest, dass in einem Pentomino jedes Quadrat mit (mindestens) einer Seite bündig an der gesamten Seite eines anderen Quadrats anliegt. Nun denkt sie über ähnliche Figuren aus fünf Quadraten nach, bei denen in mindestens einem Fall zwei Quadrate nur an einer Ecke aneinander anliegen, und zwar diagonal mit einem rechten Winkel; diese beiden Quadrate dürfen außerdem nicht an den Seiten ein und desselben dritten Quadrats anliegen. Ruth ist stolz auf ihre Erfindung und nennt diese neuen Figuren 5-Rominos, mit einem R für Ruth.

Ruth fragt sich, wie viele 5-Rominos es gibt. Ob dafür auch so eine kleine Schachtel genügen würde wie für die Pentominos? Bei 3-Rominos, also Rominos aus 3 Quadraten, ist es einfach. Es gibt genau drei unterschiedliche. Aber schon bei 4-Rominos kommt Ruth ins Grübeln. Wie viele 4-Rominos gibt es?

#### Aufgabe 5

Schreibe ein Programm, das für eine eingelesene Zahl n die Anzahl unterschiedlicher n-Rominos ermittelt und diese grafisch ausgeben kann. Figuren, die durch Verschiebung, Drehung oder Spiegelung zur Deckung zu bringen sind, gelten als identisch.

Dokumentiere die von deinem Programm berechneten Anzahlen für n von 4 bis 10 und außerdem die grafische Ausgabe des Programms für 4- und 5-Rominos.

Beispieldaten gibt es zu dieser Aufgabe nicht; als Eingabe genügt die Zahl n.

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Figuren als Mengen von Koordinaten:** Ein Romino ist eine Menge von Feldern `(x, y)`. Zwei Felder berühren sich, wenn sich ihre Koordinaten um höchstens 1 unterscheiden (Seite oder Ecke).
- **Backtracking:** Man baut eine Figur Feld für Feld auf, probiert bei jedem Schritt alle Möglichkeiten und macht den Schritt wieder rückgängig. So entstehen alle Figuren systematisch.
- **Jede Figur genau einmal erzeugen (Verfahren von Redelmeier):** Wenn man sich merkt, welche Nachbarfelder man schon betrachtet hat, taucht jede Figur nur einmal auf, solange man sie nicht verschiebt.
- **Normalisieren:** Zwei Figuren, die man verschieben kann, sind gleich. Deshalb schiebt man jede Figur so, dass ihre kleinste x- und y-Koordinate 0 ist.
- **Symmetrien:** Es gibt 8 Lagen einer Figur (4 Drehungen, jeweils gespiegelt oder nicht). Als Vertreter nimmt man die „kleinste“ Lage; dann sind gleiche Figuren auch gleich dargestellt.
- **Mengen zum Aussortieren:** In einer Menge (`set`) steht jedes Element nur einmal.

**Nützliche Funktionen und Bibliotheken**
- `set`, Tupel als Elemente von Mengen (Listen gehen nicht), `sorted(...)`, `min(...)`.
- Tupel und Listen aus Tupeln lassen sich vergleichen und sortieren: `(0, 1) < (1, 0)`. Das nutzen wir, um unter den 8 Lagen die kleinste zu wählen.

## Die Idee

1. **Alle Figuren aus `n` zusammenhängenden Feldern** erzeugen, wobei auch Ecken als Verbindung zählen. Das ergibt Pentominos *und* Rominos.
2. **Nur die gültigen behalten:** Ein Romino braucht mindestens ein Paar diagonal liegender Felder `(x, y)` und `(x+1, y+1)` (oder `(x+1, y-1)`), bei dem *keines* der beiden gemeinsamen Nachbarfelder `(x+1, y)` und `(x, y+1)` zur Figur gehört. Sonst hängen die beiden über ein drittes Quadrat an den Seiten zusammen.
3. **Gleiche Figuren zusammenfassen:** Jede Figur in alle 8 Lagen bringen, normalisieren und die kleinste Lage als Vertreter nehmen. Die Zahl der verschiedenen Vertreter ist die gesuchte Anzahl.

Ein Gitter legen wir nicht an: Wir arbeiten mit Koordinaten und lassen nur Felder zu, die an die bisherige Figur angrenzen.

## Schritt 1: Wann ist eine Figur ein Romino?

Wir prüfen für jedes Feld `(x, y)` die beiden Diagonalen nach rechts oben und nach rechts unten. Jedes diagonale Paar wird so genau einmal gefunden. Es zählt nur, wenn keines der beiden gemeinsamen Nachbarfelder zur Figur gehört.

In [ ]:
def ist_romino(figur):
    """figur: Menge von Feldern (x, y). Gibt es ein Paar, das sich nur an einer Ecke berührt, ohne gemeinsames Nachbarfeld in der Figur?"""
    for (x, y) in figur:
        for dy in [1, -1]:
            if (x + 1, y + dy) in figur:                                   # diagonaler Nachbar
                if (x + 1, y) not in figur and (x, y + dy) not in figur:   # kein drittes Quadrat dazwischen
                    return True
    return False


print(ist_romino({(0, 0), (1, 1)}))                    # zwei Felder an einer Ecke
print(ist_romino({(0, 0), (1, 1), (1, 0)}))            # das dritte Feld verbindet sie an der Seite
print(ist_romino({(0, 0), (1, 0), (2, 0), (3, 1)}))    # Stab mit Eck-Nachbar
print(ist_romino({(0, 0), (1, 0), (2, 0), (3, 0)}))    # Tetromino I: nur Seiten

True
False
True
False


Das sind `True`, `False`, `True`, `False`.

## Schritt 2: Alle zusammenhängenden Figuren mit Backtracking

Wir wollen jede Figur genau einmal erzeugen, ohne sie hinterher mit allen anderen zu vergleichen. Dazu legen wir fest: Das **Startfeld** ist das unterste Feld der linkesten Spalte, es bekommt die Koordinaten `(0, 0)`. Alle anderen Felder liegen weiter rechts (`x > 0`) oder in derselben Spalte oberhalb (`x = 0` und `y > 0`). Dadurch gibt es zu jeder Figur genau ein mögliches Startfeld, und wir erzeugen jede Figur nur in *einer* Verschiebung.

Die Rekursion `baue` hält

- `figur`: die Felder, die bisher zur Figur gehören,
- `offen`: Felder, die wir noch hinzufügen *könnten*,
- `gesehen`: Felder, die wir schon einmal in `offen` aufgenommen haben (auch wenn sie nicht gewählt wurden). Sie dürfen nie wieder auftauchen, sonst entstünde dieselbe Figur mehrfach.

Bei jedem Aufruf nehmen wir ein Feld aus `offen`, fügen es zur Figur hinzu und führen mit seinen *neuen* Nachbarn die Rekursion weiter. Danach wird das Feld **nicht** wieder angeboten: Alle Figuren, die dieses Feld enthalten, haben wir schon gebaut. So entscheiden wir für jedes Feld einmal „drin“ oder „draußen“.

In [ ]:
NACHBARN = [(1, 0), (-1, 0), (0, 1), (0, -1), (1, 1), (1, -1), (-1, 1), (-1, -1)]    # Seiten und Ecken


def erlaubt(feld):
    """Felder links vom Startfeld oder darunter sind verboten."""
    x, y = feld
    return x > 0 or (x == 0 and y >= 0)


def alle_figuren(n):
    """Alle Figuren aus n Feldern (Seiten- oder Eckenverbindung), Startfeld (0, 0), jede in genau einer Verschiebung."""
    ergebnis = []
    figur = []

    def baue(offen, gesehen):
        offen = list(offen)                      # eine Kopie, denn wir verbrauchen sie
        while len(offen) > 0:
            feld = offen.pop()
            figur.append(feld)
            if len(figur) == n:
                ergebnis.append(tuple(figur))    # fertige Figur merken
            else:
                neu = []
                for dx, dy in NACHBARN:
                    nachbar = (feld[0] + dx, feld[1] + dy)
                    if nachbar not in gesehen and erlaubt(nachbar):
                        neu.append(nachbar)
                baue(offen + neu, gesehen | set(neu))
            figur.pop()                          # Feld wieder herausnehmen (Backtracking)

    baue([(0, 0)], {(0, 0)})
    return ergebnis


for n in range(1, 7):
    print(n, len(alle_figuren(n)))

1 1
2 4
3 20
4 110
5 638
6 3832


Das sind die Zahlen der **festen** Figuren (nur Verschieben erlaubt, Drehen und Spiegeln ergibt Unterschiede). Wir erkennen die Folge 1, 4, 20, 110, 638, 3832: Pro neuem Feld wächst die Zahl auf etwa das Sechsfache. Das wird bei größerem `n` bald viel (siehe unten). Für `n = 2` sind es vier Figuren: Feld rechts, Feld oberhalb, Feld rechts oben und Feld rechts unten vom Startfeld; nur die letzten beiden sind Rominos.

Anmerkung zu `gesehen | set(neu)`: Der Aufruf bekommt eine **neue** Menge, die Menge des Aufrufers bleibt unverändert. Dadurch gelten die „gesehen“-Felder nur für die Verzweigungen unterhalb, und beim Zurückgehen ist nichts rückgängig zu machen.

## Schritt 3: Gleiche Figuren erkennen

Aus einer Figur entstehen durch Drehen und Spiegeln bis zu 8 Lagen. Wir definieren:

- `drehe(figur)`: Drehung um 90 Grad, `(x, y)` wird zu `(-y, x)`.
- `spiegle(figur)`: Spiegelung an der y-Achse, `(x, y)` wird zu `(-x, y)`.
- `normalisiere(figur)`: Verschiebt die Figur, sodass das kleinste `x` und das kleinste `y` je 0 sind, und sortiert die Felder. Das Ergebnis ist ein Tupel und damit in Mengen erlaubt.
- `kanonisch(figur)`: Alle 8 Lagen normalisieren und die **kleinste** nehmen. Tupel aus Tupeln vergleicht Python lexikografisch (wie Wörter im Wörterbuch). Zwei Figuren sind genau dann gleich, wenn sie denselben Vertreter haben.

In [ ]:
def drehe(figur):
    return [(-y, x) for (x, y) in figur]


def spiegle(figur):
    return [(-x, y) for (x, y) in figur]


def normalisiere(figur):
    kleinstes_x = min(x for (x, y) in figur)
    kleinstes_y = min(y for (x, y) in figur)
    return tuple(sorted((x - kleinstes_x, y - kleinstes_y) for (x, y) in figur))


def kanonisch(figur):
    lagen = []
    f = list(figur)
    for spiegelung in range(2):
        for drehung in range(4):
            lagen.append(normalisiere(f))
            f = drehe(f)
        f = spiegle(f)
    return min(lagen)


# Zwei Lagen derselben Figur (L-Tetromino) und ein anderes Tetromino
a = [(0, 0), (0, 1), (0, 2), (1, 0)]
b = [(0, 0), (1, 0), (2, 0), (2, 1)]
c = [(0, 0), (1, 0), (2, 0), (1, 1)]
print(kanonisch(a))
print(kanonisch(b) == kanonisch(a))
print(kanonisch(c) == kanonisch(a))

((0, 0), (0, 1), (0, 2), (1, 0))
True
False


Die beiden L-Figuren haben denselben Vertreter, das T nicht: `True` und `False`.

## Schritt 4: Rominos zählen

Jetzt setzen wir die drei Teile zusammen: alle Figuren erzeugen, die Rominos behalten, deren Vertreter in eine Menge legen. Die Größe der Menge ist die Anzahl der `n`-Rominos.

In [ ]:
def rominos(n):
    """Menge der kanonischen Vertreter aller n-Rominos."""
    vertreter = set()
    for figur in alle_figuren(n):
        if ist_romino(set(figur)):
            vertreter.add(kanonisch(figur))
    return vertreter


for n in range(2, 8):
    print('n =', n, ' Anzahl der n-Rominos:', len(rominos(n)))

n = 2  Anzahl der n-Rominos: 1
n = 3  Anzahl der n-Rominos: 3
n = 4  Anzahl der n-Rominos: 17
n = 5  Anzahl der n-Rominos: 82
n = 6  Anzahl der n-Rominos: 489
n = 7  Anzahl der n-Rominos: 2924


Für `n = 3` stimmt die Aufgabenstellung (drei Stück), für `n = 4` finden wir **17** 4-Rominos, für `n = 5` **82** 5-Rominos. Die Schachtel für Ruths Rominos müsste also deutlich größer sein als die für die zwölf Pentominos.

## Schritt 5: Grafische Ausgabe

Jedes Feld zeichnen wir als `#`, leere Felder als `.`. Damit mehrere Figuren nebeneinander in einer Reihe stehen, zeichnen wir jede Figur Zeile für Zeile (die oberste Zeile ist das größte `y`) und kleben die Zeilen mehrerer Figuren mit Abstand zusammen. Kleinere Figuren füllen wir mit Leerraum auf.

In [ ]:
def zeichne(figuren, n, pro_zeile=10):
    """Gibt die Figuren (kanonische Tupel) als Text aus, pro_zeile Stück nebeneinander."""
    for anfang in range(0, len(figuren), pro_zeile):
        gruppe = figuren[anfang:anfang + pro_zeile]
        for zeile in range(n - 1, -1, -1):              # von oben nach unten
            text = ''
            for figur in gruppe:
                breite = max(x for (x, y) in figur) + 1
                hoehe = max(y for (x, y) in figur) + 1
                for spalte in range(n):
                    if spalte >= breite or zeile >= hoehe:
                        text = text + ' '
                    elif (spalte, zeile) in figur:
                        text = text + '#'
                    else:
                        text = text + '.'
                text = text + '   '
            if text.strip() != '':                       # ganz leere Zeilen weglassen
                print(text.rstrip())
        print()


R3 = sorted(rominos(3))
print('Die', len(R3), '3-Rominos:')
zeichne(R3, 3)

Die 3 3-Rominos:
.#    #.    ..#
#.    .#    .#.
#.    #.    #..



Jede der drei Figuren hat ein Paar an einer Ecke ohne Verbindung über ein drittes Quadrat; das stimmt mit den drei Figuren der Aufgabenstellung überein.

Jetzt alle 4-Rominos:

In [ ]:
R4 = sorted(rominos(4))
print('Die', len(R4), '4-Rominos:')
zeichne(R4, 4, 9)

Die 17 4-Rominos:
.#     #.                   .#                   ..#    .#
#.     .#     .#     ..#    .#     .#.    .##    .#.    #.
#.     #.     #.     ##.    #.     #.#    #..    #..    .#
#.     #.     ##     #..    #.     #..    #..    #..    #.

              #.            ..#    .#.    ...#
#..    #..    .#     .#.    .#.    ..#    ..#.   .#.
.#.    .##    .#     .##    .#.    .#.    .#..   #.#
#.#    #..    #.     #..    #..    #..    #...   .#.



Und alle 5-Rominos (jeweils 8 pro Zeile; die Reihenfolge ist durch das Sortieren der Vertreter bestimmt):

In [ ]:
R5 = sorted(rominos(5))
print('Die', len(R5), '5-Rominos:')
zeichne(R5, 5, 8)

Die 82 5-Rominos:
.#      #.                                              .#
#.      .#      .#              .#              ..#     .#
#.      #.      #.      #..     #.      #..     ##.     #.
#.      #.      #.      #.#     ##      ##.     #..     #.
#.      #.      ##      ##.     #.      #.#     #..     #.

                ..#     #.                              .#
.#.     .##     .#.     #.      #.      #.      ##      #.
#.#     #..     #..     .#      .#      .#      .#      .#
#..     #..     #..     #.      #.      ##      #.      #.
#..     #..     #..     #.      ##      #.      #.      #.

                        #.
#..     #..     #.#     .#                              ..#
.#.     .##     .#.     .#      ..#     .#.     .##     .#.
#.#     #..     #..     #.      ##.     #.#     #..     #..
#..     #..     #..     #.      ##.     ##.     ##.     ##.

        ..#     .#.                     ..#
.#.     .#.     ..#     ..#     ...#    ..#     ..#.    ..##
##.     ##.     #

## Schritt 6: Das Programm für beliebiges n

Als Zusammenfassung (die Aufgabe will eine eingelesene Zahl `n`): Die Funktion `romino_programm` gibt die Anzahl aus und auf Wunsch die Figuren. Im Notebook rufen wir sie mit festen Zahlen auf; in einem Skript würde man `n = int(input('n: '))` verwenden.

In [ ]:
def romino_programm(n, zeige=False):
    figuren = sorted(rominos(n))
    print('Es gibt', len(figuren), str(n) + '-Rominos.')
    if zeige:
        zeichne(figuren, n)


romino_programm(4)
romino_programm(5)
romino_programm(8)

Es gibt 17 4-Rominos.
Es gibt 82 5-Rominos.
Es gibt 18406 8-Rominos.


## Prüfung: Anzahlen n = 2 bis 8 und die Tabelle der Lösungshinweise

In den Lösungshinweisen steht diese Tabelle der Anzahlen: 2: 1, 3: 3, 4: 17, 5: 82, 6: 489, 7: 2924, 8: 18406, 9: 116883, 10: 753905, 11: 4899488. Wir vergleichen unser Programm mit den Werten bis 8. Dabei zählen wir auch, wie viele feste Figuren dazu erzeugt werden mussten.

In [ ]:
TABELLE = {2: 1, 3: 3, 4: 17, 5: 82, 6: 489, 7: 2924, 8: 18406}
for n in range(2, 9):
    print('n =', n, ' feste Figuren:', len(alle_figuren(n)), ' Rominos:', len(rominos(n)), ' Tabelle:', TABELLE[n])

n = 2  feste Figuren: 4  Rominos: 1  Tabelle: 1
n = 3  feste Figuren: 20  Rominos: 3  Tabelle: 3
n = 4  feste Figuren: 110  Rominos: 17  Tabelle: 17
n = 5  feste Figuren: 638  Rominos: 82  Tabelle: 82
n = 6  feste Figuren: 3832  Rominos: 489  Tabelle: 489
n = 7  feste Figuren: 23592  Rominos: 2924  Tabelle: 2924
n = 8  feste Figuren: 147941  Rominos: 18406  Tabelle: 18406


Alle Werte stimmen mit der Tabelle überein.

**Die Zahlen für n = 9 und n = 10.** Mit demselben Programm erhalten wir bei `n = 9` die Anzahl **116883** und bei `n = 10` die Anzahl **753905**, ebenfalls wie in der Tabelle. Diese beiden Läufe sind im Notebook nicht ausgeführt, denn sie brauchen mit diesem einfachen Python-Programm Minuten (bei `n = 10` sind allein über sechs Millionen feste Rominos zu prüfen, jedes wird in 8 Lagen umgerechnet) und im Browser noch deutlich länger. Probiere es bei Interesse aus: `romino_programm(9)`.

**Zur Dokumentation (Aufgabe):**

| n | 4 | 5 | 6 | 7 | 8 | 9 | 10 |
|---|---|---|---|---|---|---|---|
| Anzahl | 17 | 82 | 489 | 2924 | 18406 | 116883 | 753905 |

## Eine Überraschung bei n = 7

Man könnte denken, dass Rominos genau die Figuren sind, die *nicht* allein durch Seiten zusammenhängen. Dann müsste die Zahl der 7-Rominos der Zahl der Figuren mit Seiten- oder Eckenverbindung minus der Zahl der 108 Heptominos entsprechen (3031 − 108 = 2923, wenn man diese bekannten Zahlen zugrunde legt). Die Tabelle sagt aber 2924. Wo liegt der Unterschied? Eine Figur kann **durch Seiten zusammenhängen** und trotzdem ein Eckenpaar haben, das nicht über ein drittes Quadrat verbunden ist, wenn die beiden Felder auf einem Ring weit voneinander entfernt sind. Wir suchen unter den 7-Rominos die, deren Felder alle über Seiten zusammenhängen:

In [ ]:
def seitenverbunden(figur):
    """Hängen alle Felder über gemeinsame Seiten zusammen? (Flächenfüllen ab dem ersten Feld)"""
    figur = set(figur)
    start = next(iter(figur))
    erreicht = {start}
    neue = [start]
    while len(neue) > 0:
        x, y = neue.pop()
        for nachbar in [(x + 1, y), (x - 1, y), (x, y + 1), (x, y - 1)]:
            if nachbar in figur and nachbar not in erreicht:
                erreicht.add(nachbar)
                neue.append(nachbar)
    return len(erreicht) == len(figur)


sonderfaelle = [f for f in sorted(rominos(7)) if seitenverbunden(f)]
print(len(sonderfaelle), 'Romino(s) mit n = 7, das auch durch Seiten zusammenhängt:')
zeichne(sonderfaelle, 7)

1 Romino(s) mit n = 7, das auch durch Seiten zusammenhängt:
##.
#.#
###



Das ist der Rand eines 3 × 3-Quadrats, bei dem eine Ecke fehlt: ein Ring mit Lücke. An der Lücke berühren sich zwei Felder nur an einer Ecke, und weder die Lücke noch das leere Mittelfeld gehören zur Figur. Die Figur erfüllt die Aufgabenstellung wörtlich und ist deshalb ein 7-Romino. Bei kleineren `n` gibt es so etwas nicht, weil ein solcher Ring mindestens 7 Felder braucht.

## Für Neugierige: schneller zählen

- Die Lösungshinweise erwähnen die Optimierung, die wir schon eingebaut haben: Das Startfeld ist das unterste Feld der linkesten Spalte (Redelmeier). Damit entsteht jede Figur nur einmal pro Lage statt `n`-mal.
- **Speicher:** Statt erst alle Figuren zu sammeln, kann man jede sofort prüfen und nur den Vertreter speichern. Besser als in `rominos` (dort bleibt zwar nur ein Vertreter pro Figur im Speicher, aber `alle_figuren` sammelt zuerst alle festen Figuren) wäre es, in `baue` direkt zu prüfen und nur Rominos zu speichern.
- Noch weniger Speicher braucht man, wenn man eine Figur nur zählt, wenn sie gleich ihrem eigenen Vertreter ist (`normalisiere(figur) == kanonisch(figur)`); Figuren mit Symmetrie muss man dann mit der richtigen Gewichtung zählen (Lemma von Burnside). Das ist ein guter Ausblick, aber nicht nötig.
- Für die Anzahl der Polyominos gibt es viele sehr viel schnellere Verfahren (siehe Wikipedia, „Polyomino“); sie lassen sich meist auf Rominos übertragen.

## Zum Schluss: Rominos mit MiniZinc?

Wir haben geprüft, ob sich die Aufgabe sinnvoll mit MiniZinc lösen lässt. Ergebnis: nein, jedenfalls nicht besser als mit dem Programm oben; ein Modell haben wir deshalb nicht aufgenommen.

- MiniZinc löst Aufgaben der Art „finde eine Belegung (oder die beste), die alle Bedingungen erfüllt“. Hier ist gesucht, **wie viele** Figuren es bis auf Verschieben, Drehen und Spiegeln gibt, und sie sollen aufgelistet werden. Alle Lösungen aufzählen geht, aber die Gleichheit bis auf Symmetrie müsste als Bedingung formuliert werden („jede Lösung ist die kleinste ihrer 8 Lagen“), und das ist ein umständliches Modell mit vielen Hilfsvariablen.
- Auch der Zusammenhang der Figur und die Rominobedingung („es gibt ein Eckenpaar ohne gemeinsames Nachbarquadrat“) sind in MiniZinc nur über Hilfskonstruktionen (Abstandsvariablen, `exists` über das Gitter) auszudrücken. Diese Hilfsvariablen müssen pro Figur eindeutig bestimmt sein, sonst wird mehrfach gezählt.
- Bei `n = 10` müsste der Solver 753 905 Lösungen einzeln erzeugen, nach dem Aufzählen der Lösungen mit den Hilfsvariablen sogar noch mehr.

Das Aufzählen von Strukturen mit Backtracking ist hier die natürliche Lösung.